# Exact solver results per graph

Solves the Bd Moran process exactly on a list of graphs with Hila's solver (`rust/exact`), one
multi-core LSF job per graph, then prints the averages over start nodes and draws each graph
colored by each start node's fixation probability, fixation time and absorption time.

The solver reads one JSON line, `{"graph": adjacency lists, "fitness": r, "threads": cores}`, and
prints, for each node `i`, the result of a single mutant starting at `i`: `prob` (fixation
probability), `time` (absorption time, over all runs) and `ctime` (fixation time, conditional on
fixation). Times are in Moran steps, null steps included, as in `simulate()` and `theory`.

**To use:** list graphs in section 1 and run all cells. The submit cell submits only the graphs that
are neither solved nor queued; re-run sections 3 and 4 as jobs finish.

**Saved** per graph under `EXACT_ROOT`, in a folder named `<graph name>_<WL hash>_r<r>`:
`input.json` (exactly what the solver read), `output.json` and `job.log`. A solved graph is read
from disk, not solved again; delete its folder to solve it again.

In [ ]:
%cd /home/labs/pilpel/matanyaw/moran-process
import json, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from moran_graphs import PopulationGraph, theory
from moran_graphs.hpc import ALIVE, DATA_ROOT
from moran_graphs.properties import wl_hash

CRATE = Path("rust/exact").resolve()
SOLVER = CRATE / "target/release/numeric-solver"
EXACT_ROOT = DATA_ROOT / "exact-solutions"
# Incremental, a no-op when the binary is newer than the Rust sources (~6 s otherwise).
subprocess.run([Path.home() / ".cargo/bin/cargo", "build", "--release", "--bin", "numeric-solver"], cwd=CRATE, check=True)

## 1. Graphs

Any factory works if the graph is undirected (directed graphs are unverified in this solver, and an
earlier version was wrong on the directed cycle) and has 10 to 63 nodes (N = 8 never converged).
Edge weights are not read. The defaults all have N = 15, so they compare directly, and each solves
in under a second. The solver keeps three f32 per mutant set, 12 B x 2^N (N = 24 is 200 MB,
N = 28 is 3 GB, N = 30 is 13 GB). On 16 cores N = 24 took 45 s, and the time grows about 2.6x per
extra node (N = 28 about 35 min, N = 30 about 4 h).

In [ ]:
GRAPHS = [
    PopulationGraph.mammalian_lung_graph(depth=3, branching_factor=2),
    # PopulationGraph.mammalian_lung_graph(depth=3, branching_factor=2, directed=True),
    PopulationGraph.fish_graph(n_rods=3, l_rod=2, lamellae=1),
    # PopulationGraph.fish_graph(n_rods=3, l_rod=2, lamellae=1, directed=True),
    PopulationGraph.avian_graph(n_parabronchi=4, l_parabronchi=3),
    # PopulationGraph.avian_graph(n_parabronchi=4, l_parabronchi=3, directed=True),

    PopulationGraph.cycle_graph(15),
    PopulationGraph.cycle_graph(15, directed=True),
    PopulationGraph.line_graph(15),
    PopulationGraph.grid_graph(3, 5),
    PopulationGraph.complete_graph(15),
]
R = 1.1
CORES = 16
QUEUE = "gsla-cpu"  # the lab queue is never preempted, so a long solve never restarts

assert len({g.name for g in GRAPHS}) == len(GRAPHS), "two graphs share a name: set g.name"
for g in GRAPHS:
    assert not g.graph.is_directed(), f"{g.name}: the solver is unverified on directed graphs"
    assert 10 <= g.graph.number_of_nodes() <= 63, f"{g.name}: the solver needs 10 to 63 nodes"
pd.DataFrame([(g.name, g.graph.number_of_nodes(), g.graph.number_of_edges()) for g in GRAPHS],
             columns=["graph", "N", "E"])

## 2. Submit

Safe to re-run: for each graph, it submits only when there is no result and no job for that graph
and r is queued or running. A failed job is left for you to read, not resubmitted.

In [ ]:
def result_dir(graph):
    # The WL hash is the graph's identity; names alone repeat (random_graph's name has no seed).
    return EXACT_ROOT / f"{graph.name}_{wl_hash(graph.csr())}_r{R}"


def job_state(folder):
    # bjobs lists only unfinished jobs; once LSF drops one, the files say how it ended.
    listed = subprocess.run(["bjobs", "-noheader", "-o", "stat", "-J", f"exact_{folder.name}"],
                            capture_output=True, text=True).stdout.split()
    if listed:
        return listed[-1]
    if (folder / "output.json").exists():
        return "DONE"
    return "EXIT" if (folder / "job.log").exists() else "not submitted"


def submit(graph, folder):
    W = graph.csr()
    n_nodes = W.shape[0]
    # Solver node i is row i of W, the same node order nx.draw colors in.
    adjacency = [row.tolist() for row in np.split(W.indices, W.indptr[1:-1])]
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "input.json").write_text(json.dumps({"graph": adjacency, "fitness": R, "threads": CORES}) + "\n")
    memory_mb = 12 * 2**n_nodes / 2**20 * 1.2 + 512  # three f32 per mutant set, plus headroom
    script = f"""#!/bin/bash
#BSUB -J exact_{folder.name}
#BSUB -q {QUEUE}
#BSUB -n {CORES}
#BSUB -R "span[hosts=1] rusage[mem={int(memory_mb / CORES) + 1}]"
#BSUB -oo {folder}/job.log
set -euo pipefail
cd {folder}
echo "ran on $(hostname): $(nproc) cores bound, $(lscpu | sed -n 's/^Model name: *//p')"
# Write, then rename: an output.json exists only once the solve has finished.
{SOLVER} < input.json > output.json.tmp
mv output.json.tmp output.json
"""
    # Script on stdin: the WEXAC job starter re-evaluates $vars and quotes on the bsub line.
    out = subprocess.run(["bsub"], input=script, text=True, capture_output=True, check=True)
    print(f"{graph.name}: {out.stdout.strip()}")


for g in GRAPHS:
    state = job_state(result_dir(g))
    if state == "DONE":
        print(f"{g.name}: solved")
    elif state in ALIVE:
        print(f"{g.name}: job {state}")
    elif state == "EXIT":
        print(f"{g.name}: the last job failed, see {result_dir(g) / 'job.log'}; delete that log to resubmit")
    else:
        submit(g, result_dir(g))

## 3. Averages over start nodes

One mutant on a uniformly random node. The fixation time is conditional on fixation, so each start
node weighs by its fixation probability, `sum(p_i * t_i) / sum(p_i)`, which is what `simulate()`
measures. `fixation prob / complete` compares with the complete graph of the same N (from `theory`): at
r > 1, above 1 marks an amplifier of selection, below 1 a suppressor.

In [ ]:
def summarize(per_node):
    fixation_prob = per_node.prob.mean()
    return {
        "N": len(per_node),
        "fixation_prob": fixation_prob,
        "fixation prob / complete": fixation_prob / theory.complete(len(per_node), R)[0],
        "absorption time": per_node.time.mean(),
        "fixation time": np.average(per_node.ctime, weights=per_node.prob),
    }


per_node = {}
for g in GRAPHS:
    output = result_dir(g) / "output.json"
    if output.exists():
        per_node[g.name] = pd.read_json(output)  # row i: one mutant starting at node i
    else:
        print(f"{g.name}: not solved yet, job {job_state(result_dir(g))}")
summary = pd.DataFrame.from_dict({name: summarize(df) for name, df in per_node.items()}, orient="index")
summary.round({"fixation_prob": 5, "fixation prob / complete": 3, "absorption time": 1, "fixation time": 1})

## 4. Per start node

Each panel has its own color scale. On node-symmetric graphs (cycle, complete) the solver's values
differ only by f32 noise, about 1e-6 of the value, so every scale spans at least 0.1% of its mean:
noise stays one color instead of filling the whole colormap.

In [ ]:
PANELS = [("prob", "fixation probability", "RdYlGn"),
          ("ctime", "fixation time (steps, given fixation)", "viridis"),
          ("time", "absorption time (steps, all runs)", "viridis")]


def color_norm(values):
    center = (values.max() + values.min()) / 2
    span = max(values.max() - values.min(), 1e-3 * values.mean())
    return Normalize(center - span / 2, center + span / 2)


def draw_per_node(graph, per_node):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (column, label, cmap) in zip(axes, PANELS):
        norm = color_norm(per_node[column])
        graph.draw(ax=ax, descriptive=False, node_color=plt.get_cmap(cmap)(norm(per_node[column])),
                   node_size=150, title=label)
        fig.colorbar(ScalarMappable(norm, cmap), ax=ax, shrink=0.8)
    fig.suptitle(f"{graph.name}, r = {R}, by start node", fontsize=16)
    return fig


for g in GRAPHS:
    if g.name in per_node:
        draw_per_node(g, per_node[g.name])
        plt.show()